# Mentee-Mentor Matching Analysis


## 1. Import Libraries

In [1]:
import pandas as pd
import re
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

import sys
print(sys.executable)


Libraries imported successfully!
/Users/maltejuuljohansen/Desktop/Conflux/.venv/bin/python3


## API Configuration (OpenAI)

In [2]:
import os
from pathlib import Path

# Load `.env` if python-dotenv is installed. Tries (1) kernel working directory, (2) this project folder,
# because Jupyter/Cursor sometimes start the kernel with cwd != the folder where the notebook lives.
_DOTENV_INSTALLED = False
try:
    from dotenv import load_dotenv

    _DOTENV_INSTALLED = True
    _CONFLUX_ROOT = Path("/Users/maltejuuljohansen/Desktop/Conflux")
    _env_paths = [Path.cwd() / ".env", _CONFLUX_ROOT / ".env"]
    _loaded_path = None
    for _p in _env_paths:
        if _p.is_file():
            load_dotenv(_p, override=False)
            _loaded_path = _p
            break
except ImportError:
    print("Install python-dotenv so `.env` is read automatically:  pip install python-dotenv")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY and _DOTENV_INSTALLED:
    for _p in [Path.cwd() / ".env", Path("/Users/maltejuuljohansen/Desktop/Conflux/.env")]:
        if _p.is_file():
            _first = next((ln.strip() for ln in _p.read_text(encoding="utf-8").splitlines() if ln.strip()), "")
            if _first and not _first.startswith("OPENAI_API_KEY="):
                print(
                    f"Found `.env` at {_p}, but the first non-empty line must start with OPENAI_API_KEY=\n"
                    "Example:  OPENAI_API_KEY=sk-...   (no spaces around =)"
                )
            break

if OPENAI_API_KEY:
    print("OpenAI API key loaded (environment and/or .env).")
    from openai import OpenAI

    openai_client = OpenAI(api_key=OPENAI_API_KEY)
else:
    openai_client = None
    print(
        "OPENAI_API_KEY not found. Before running OpenAI cells, do one of the following:\n"
        "  • In the terminal you use to start Jupyter:  export OPENAI_API_KEY='your-key-here'\n"
        "  • Or create `.env` in this folder with:       OPENAI_API_KEY=your-key-here\n"
        "    (requires: pip install python-dotenv)\n"
        "  • In Cursor/VS Code you can also set the variable in the integrated terminal or in Run/Debug env settings."
    )

# OpenAI quality scoring enabled.
USE_OPENAI_QUALITY = True

USE_OPENAI_SHARED_KEYWORDS = True
OPENAI_KEYWORD_MODEL = "gpt-4.1-mini"

# WordPress: "wordpress_api" = same get_posts call as Make (WP Webhooks integromat endpoint).
# Fallback: "json_file" reads a saved Make export (wordpress_posts.json).
# Use json_file locally if live API returns HTTP 455 (Simply.com firewall).
DATA_SOURCE = "wordpress_api"
WORDPRESS_BASE_URL = "https://conflux.dk"
WORDPRESS_JSON_PATH = _CONFLUX_ROOT / "wordpress_posts.json"
WP_POST_TYPE = "ex_team"
WP_TAXONOMY_YEAR = "2025"  # extp_loc slug filter (same as Make tax_query)
MENTOR_POST_STATUS = "publish"


OpenAI API key loaded (environment and/or .env).


## 2. Load Data from WordPress


In [3]:
import json
import re
import requests

QUESTION_PROFILE = "Academic and professional background"
QUESTION_MOTIVATION = "Motivation"
QUESTION_COMMENTS = "Comments"


def wp_meta_first(post, key):
    vals = post.get("meta_data", {}).get(key)
    if not vals:
        return ""
    return str(vals[0]).strip() if isinstance(vals, list) else str(vals).strip()


def wp_strip_html(text):
    text = re.sub(r"<br\s*/?>", "\n", str(text or ""), flags=re.I)
    text = re.sub(r"<[^>]+>", " ", text)
    return " ".join(text.split())


def wp_parse_sections(post_content):
    content = str(post_content or "")
    parts = re.split(r"<h2>\s*([^<]+?)\s*</h2>", content, flags=re.I)
    intro = wp_strip_html(parts[0]) if parts else ""
    sections = {}
    for i in range(1, len(parts) - 1, 2):
        header = parts[i].strip().lower()
        body = wp_strip_html(parts[i + 1])
        sections[header] = body
    return intro, sections.get("motivation", ""), sections.get("comments", "")


def wp_load_posts_from_json(path):
    raw = json.loads(Path(path).read_text(encoding="utf-8"))
    if isinstance(raw, list) and raw and isinstance(raw[0], dict) and "data" in raw[0]:
        return raw[0]["data"]["posts"]
    if isinstance(raw, dict) and "data" in raw and "posts" in raw["data"]:
        return raw["data"]["posts"]
    if isinstance(raw, dict) and "posts" in raw:
        return raw["posts"]
    if isinstance(raw, list):
        return raw
    raise ValueError(f"Unrecognized WordPress JSON shape in {path}")


def wp_make_get_posts_arguments(post_type=WP_POST_TYPE, year=None):
    """Same arguments JSON as Make WP Webhooks module (get_posts)."""
    year = year or WP_TAXONOMY_YEAR
    return {
        "post_type": post_type,
        "posts_per_page": -1,
        "post_status": "any",
        "tax_query": [
            {"taxonomy": "extp_loc", "field": "slug", "terms": year}
        ],
        "load_meta": "yes",
    }


def wp_extract_posts_from_response(data):
    if isinstance(data, list) and data and isinstance(data[0], dict) and "data" in data[0]:
        data = data[0]["data"]
    if isinstance(data, dict) and "data" in data and isinstance(data["data"], dict):
        inner = data["data"]
        if "posts" in inner:
            return inner["posts"]
    if isinstance(data, dict) and "posts" in data:
        return data["posts"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unexpected WordPress API response: {str(data)[:500]}")


def wp_fetch_posts_from_api(base_url, api_key, post_type=WP_POST_TYPE, year=None):
    """Replicate Make: WP Webhooks integromat endpoint -> action get_posts."""
    url = f"{base_url.rstrip('/')}/"
    # Simply.com WAF returns HTTP 455 for default python-requests (no User-Agent).
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
        "Accept": "application/json, text/plain, */*",
    }
    params = {"wpwhpro_action": "integromat"}
    payload = {
        "wpwhpro_api_key": api_key,
        "action": "get_posts",
        "arguments": json.dumps(wp_make_get_posts_arguments(post_type, year)),
        "load_meta": "yes",
    }
    resp = requests.post(url, params=params, data=payload, headers=headers, timeout=120)
    if resp.status_code == 455:
        raise RuntimeError(
            "WordPress returned HTTP 455 (Simply.com WAF blocked the request). "
            "Ensure wp_fetch_posts_from_api sends a browser User-Agent header. "
            "Temporary fallback: set DATA_SOURCE = \"json_file\"."
        )
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, dict) and data.get("success") is False:
        raise RuntimeError(f"WordPress API error: {data.get('msg', data)}")
    return wp_extract_posts_from_response(data)


def wp_post_to_mentee_row(post):
    intro, motivation, comments = wp_parse_sections(post.get("post_content", ""))
    return {
        "wp_id": post.get("ID"),
        "navn": str(post.get("post_title", "")).strip(),
        "Completion time": post.get("post_date"),
        "post_status": post.get("post_status"),
        "studie": wp_meta_first(post, "position"),
        "onske_1": wp_meta_first(post, "mentee_priority_1"),
        "onske_2": wp_meta_first(post, "mentee_priority_2"),
        "onske_3": wp_meta_first(post, "mentee_priority_3"),
        "video_link": wp_meta_first(post, "video_link"),
        "question_profile": QUESTION_PROFILE,
        "answer_profile": intro,
        "question_motivation": QUESTION_MOTIVATION,
        "answer_motivation": motivation,
        "question_comments": QUESTION_COMMENTS,
        "answer_comments": comments,
        "tekst": f"{intro} {motivation}".strip(),
    }


def wp_post_to_mentor_row(post):
    return {
        "wp_id": post.get("ID"),
        "navn": str(post.get("post_title", "")).strip(),
        "tekst": wp_strip_html(post.get("post_content", "")),
        "post_status": post.get("post_status"),
    }


def load_wordpress_team_posts():
    api_key = os.getenv("WPWH_PRO_API_KEY") or os.getenv("WPWH_WEBHOOK_API_KEY")
    if DATA_SOURCE == "wordpress_api":
        if not api_key:
            raise RuntimeError(
                "DATA_SOURCE=wordpress_api requires WPWH_PRO_API_KEY in .env "
                "(integromat endpoint key from WP Webhooks Settings)."
            )
        posts = wp_fetch_posts_from_api(WORDPRESS_BASE_URL, api_key)
        print(
            f"Fetched {len(posts)} posts from WordPress (get_posts, extp_loc={WP_TAXONOMY_YEAR})"
        )
    else:
        posts = wp_load_posts_from_json(WORDPRESS_JSON_PATH)
        print(f"Loaded {len(posts)} posts from {WORDPRESS_JSON_PATH}")
    return posts


posts = load_wordpress_team_posts()
mentee_posts = [p for p in posts if wp_meta_first(p, "type") == "mentee"]
mentor_posts = [
    p for p in posts
    if wp_meta_first(p, "type") == "mentor" and p.get("post_status") == MENTOR_POST_STATUS
]

mentees = pd.DataFrame([wp_post_to_mentee_row(p) for p in mentee_posts])
mentorer = pd.DataFrame([wp_post_to_mentor_row(p) for p in mentor_posts])

mentees_renset = (
    mentees
    .sort_values("Completion time")
    .drop_duplicates(subset=["navn"], keep="last")
    .copy()
)
mentorer_renset = mentorer.copy()

print(f"Mentees: {len(mentees)} raw → {len(mentees_renset)} after dedup")
print(f"Mentors (status={MENTOR_POST_STATUS}): {len(mentorer_renset)}")


Fetched 393 posts from WordPress (get_posts, extp_loc=2025)
Mentees: 272 raw → 266 after dedup
Mentors (status=publish): 114


## 3. Generate Embeddings

OpenAI `text-embedding-3-large` (run **API Configuration** first).


In [4]:
import time
import numpy as np
from sklearn.preprocessing import normalize

openai_client = globals().get("openai_client")
if openai_client is None:
    raise RuntimeError(
        "openai_client is not initialized. Run the API Configuration cell and set OPENAI_API_KEY."
    )

OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"
EMBED_BATCH_SIZE = 64


def _clean_text_for_embed(t):
    if t is None or (isinstance(t, float) and pd.isna(t)):
        return " "
    s = str(t).strip()
    return s if s else " "


def embed_texts_openai(texts, label="items"):
    vectors = []
    n = len(texts)
    for start in range(0, n, EMBED_BATCH_SIZE):
        end = min(start + EMBED_BATCH_SIZE, n)
        batch = [_clean_text_for_embed(t) for t in texts[start:end]]
        last_error = None
        for attempt in range(1, 4):
            try:
                resp = openai_client.embeddings.create(
                    model=OPENAI_EMBEDDING_MODEL,
                    input=batch,
                )
                ordered = sorted(resp.data, key=lambda d: d.index)
                vectors.extend([d.embedding for d in ordered])
                print(f"  {label}: {end}/{n}")
                break
            except Exception as exc:
                last_error = exc
                if attempt < 3:
                    time.sleep(1.5 * attempt)
                else:
                    raise RuntimeError(
                        f"OpenAI embedding request failed after 3 attempts: {last_error}"
                    ) from last_error
    arr = np.asarray(vectors, dtype=np.float32)
    return normalize(arr, norm="l2")


print(f"Encoding mentors via {OPENAI_EMBEDDING_MODEL}...")
mentor_embeddings = embed_texts_openai(mentorer_renset["tekst"].tolist(), label="mentors")

print(f"Encoding mentees via {OPENAI_EMBEDDING_MODEL}...")
mentee_embeddings = embed_texts_openai(mentees_renset["tekst"].tolist(), label="mentees")

print("Computing cosine similarity for retrieval...")
similarities = cosine_similarity(mentee_embeddings, mentor_embeddings)
print(f"Similarity matrix: {similarities.shape}")

Encoding mentors via text-embedding-3-large...


  mentors: 64/114


  mentors: 114/114
Encoding mentees via text-embedding-3-large...


  mentees: 64/266


  mentees: 128/266


  mentees: 192/266


  mentees: 256/266


  mentees: 266/266
Computing cosine similarity for retrieval...
Similarity matrix: (266, 114)


## 4. Name Mappings and Helper Functions


In [5]:
import time

mentee_names = mentees_renset["navn"].tolist()
mentor_names = mentorer_renset["navn"].tolist()
mentor_name_to_index = {navn: i for i, navn in enumerate(mentor_names)}

SHARED_KEYWORDS_SYSTEM = """You extract shared academic/professional profile keywords from a mentee-mentor pair.

Return ONLY a comma-separated list of 3-8 concrete overlap terms (study fields, research themes, tools, methods, industries, roles).
Use lowercase. Use underscores for multi-word terms (e.g. machine_learning, supply_chain).
Exclude generic filler (student, mentor, mentee, experience, passion, dtu, denmark, university, career, guidance).
If there is almost no meaningful overlap, return an empty string.
No explanation. No numbering."""


def find_shared_terms(tekst1, tekst2, top_n=8):
    client = globals().get("openai_client")
    if client is None or not globals().get("USE_OPENAI_SHARED_KEYWORDS", True):
        return ""

    model = globals().get("OPENAI_KEYWORD_MODEL", "gpt-4.1-mini")
    user_prompt = (
        f"Mentee profile text:\n{str(tekst1)[:2500]}\n\n"
        f"Mentor profile text:\n{str(tekst2)[:2500]}"
    )
    last_error = None
    for attempt in range(1, 4):
        try:
            response = client.chat.completions.create(
                model=model,
                temperature=0,
                max_tokens=120,
                messages=[
                    {"role": "system", "content": SHARED_KEYWORDS_SYSTEM},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = (response.choices[0].message.content or "").strip()
            terms = []
            for part in raw.split(","):
                token = part.strip().lower().replace("-", "_").replace(" ", "_")
                token = re.sub(r"[^a-z0-9_]", "", token)
                if len(token) >= 2:
                    terms.append(token)
            return ", ".join(terms[:top_n])
        except Exception as exc:
            last_error = exc
            if attempt < 3:
                time.sleep(1.0 * attempt)
            else:
                print(f"OpenAI shared-keywords failed ({last_error})")
                return ""


mentee_text_map = dict(zip(mentees_renset["navn"], mentees_renset["tekst"]))
mentor_text_map = dict(zip(mentorer_renset["navn"], mentorer_renset["tekst"]))

print("Setup complete")


Setup complete


## 5. Build Match Candidates


In [6]:
all_match_candidates = []
for i, mentee_name in enumerate(mentee_names):
    for mentor_idx, mentor_name in enumerate(mentor_names):
        all_match_candidates.append({
            "mentee": mentee_name,
            "mentor": mentor_name,
            "similarity_score": round(float(similarities[i, mentor_idx]), 4)
        })

all_match_candidates = pd.DataFrame(all_match_candidates)
print(f"Built {len(all_match_candidates)} mentee-mentor candidates")


Built 30324 mentee-mentor candidates


## 6. Video score (for OpenAI quality blend)


In [7]:
# Video flag and score (blended into OpenAI quality_score_final in the next section)
mentees_renset["video"] = mentees_renset["video_link"].astype(str).str.contains("https", case=False, na=False).astype(int)
mentees_renset["video_score"] = mentees_renset["video"] * 10

visible_score_cols = [
    "video_score"
]
existing_score_cols = [col for col in visible_score_cols if col in all_match_candidates.columns]
if existing_score_cols:
    all_match_candidates = all_match_candidates.drop(columns=existing_score_cols)

all_match_candidates = all_match_candidates.merge(
    mentees_renset[[
        "navn",
        "video_score"
    ]],
    left_on="mentee",
    right_on="navn",
    how="left"
).drop(columns=["navn"])


## 7. OpenAI Quality Score


In [8]:
import re
import time

# Set in the API Configuration cell; avoids NameError if that cell was not run yet.
openai_client = globals().get("openai_client")

USE_OPENAI_QUALITY = True
OPENAI_QUALITY_MODEL = "gpt-4.1-mini"

QUALITY_SYSTEM_PROMPT = """You are a strict admissions evaluator.

Your job is NOT to be helpful or encouraging. Your job is to judge quality.

You will see three survey questions (as provided) and the student's answers. Give ONE holistic score from 0 to 10 for the combined written application (all three answers together).

Score using the criteria below:

1. RELEVANCE (0–4 points)
- Does each answer address its stated question?
- Or is it generic, off-topic, or mostly boilerplate?

2. CONCRETENESS (0–3 points)
- Does the student give specific experiences, examples, or goals?
- Or is it vague and abstract?

3. DEPTH (0–3 points)
- Does the writing show reflection, motivation, or substance?
- Or is it superficial?

IMPORTANT RULES:
- Use ONLY the questions and answers shown in the user message. Do not invent facts about the student.
- Judge substance in whatever language the student wrote in (do not penalize for Danish vs English).
- Do NOT treat longer answers as automatically better; reward specificity and substance, not word count.
- Be strict. Most combined applications should NOT score above 7.
- A score of 9–10 is extremely rare and requires excellent depth and specificity across the answers that matter.
- Penalize vague, generic, or repetitive answers heavily.
- Be objective and critical.

Return ONLY a single number from 0 to 10.
No explanation. No text."""

def _extract_score(raw_text, prefer_score_line=False):
    text = str(raw_text).strip().replace(",", ".")
    if prefer_score_line:
        m = re.search(r"(?i)score:\s*(-?\d+(?:\.\d+)?)", text)
        if m:
            score = float(m.group(1))
            return max(0.0, min(10.0, round(score, 2)))
    match = re.search(r"-?\d+(?:\.\d+)?", text)
    if not match:
        raise ValueError(f"No numeric score found in response: {raw_text}")
    score = float(match.group(0))
    return max(0.0, min(10.0, round(score, 2)))


def _openai_numeric_score(system_prompt, user_prompt, model_name, retries=3, prefer_score_line=False):
    if openai_client is None:
        raise RuntimeError(
            "openai_client is not initialized. Set OPENAI_API_KEY (see the API Configuration cell)."
        )

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = openai_client.chat.completions.create(
                model=model_name,
                temperature=0,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = response.choices[0].message.content
            return _extract_score(raw, prefer_score_line=prefer_score_line)
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(1.5 * attempt)
            else:
                raise RuntimeError(f"OpenAI scoring failed after {retries} attempts: {exc}") from exc


def build_quality_prompt(row):
    q_profile = str(row.get("question_profile", QUESTION_PROFILE)).strip()
    q_motivation = str(row.get("question_motivation", QUESTION_MOTIVATION)).strip()
    q_comments = str(row.get("question_comments", QUESTION_COMMENTS)).strip()
    a_profile = str(row.get("answer_profile", "") or "").strip()
    a_motivation = str(row.get("answer_motivation", "") or "").strip()
    a_comments = str(row.get("answer_comments", "") or "").strip()

    return f"""Score the following application (three question-and-answer pairs). Use each question to judge relevance of the answer directly below it.

--- Question 1 ---
{q_profile}

Answer 1:
{a_profile}

--- Question 2 ---
{q_motivation}

Answer 2:
{a_motivation}

--- Question 3 ---
{q_comments}

Answer 3:
{a_comments}""".strip()


def score_quality_openai(row):
    prompt = build_quality_prompt(row)
    return _openai_numeric_score(QUALITY_SYSTEM_PROMPT, prompt, OPENAI_QUALITY_MODEL)


if USE_OPENAI_QUALITY:
    mentees_renset["quality_score_openai"] = mentees_renset.apply(score_quality_openai, axis=1)
    # 90% OpenAI text quality, 10% video_score.
    mentees_renset["quality_score_final"] = (
        mentees_renset["quality_score_openai"] * 0.9 + mentees_renset["video_score"] * 0.1
    ).round(2)
else:
    raise RuntimeError("USE_OPENAI_QUALITY must be True for this workflow.")

visible_score_cols = [
    "quality_score_final",
    "quality_score_openai",
    "video_score"
]
existing_score_cols = [col for col in visible_score_cols if col in all_match_candidates.columns]
if existing_score_cols:
    all_match_candidates = all_match_candidates.drop(columns=existing_score_cols)

all_match_candidates = all_match_candidates.merge(
    mentees_renset[[
        "navn",
        "quality_score_final",
        "quality_score_openai",
        "video_score"
    ]],
    left_on="mentee",
    right_on="navn",
    how="left"
).drop(columns=["navn"])

print("OpenAI quality scoring complete.")


OpenAI quality scoring complete.


## 8. Total Score


In [9]:
# Formula: Total Score = (Cosine * 50%) + (Quality * 30%) + (Preference * 20%)
# quality_score_final = 90% OpenAI text score + 10% video_score

all_match_candidates["cosine_norm"] = (all_match_candidates["similarity_score"] * 10).round(2)
all_match_candidates["retrieval_rank"] = (
    all_match_candidates.groupby("mentee")["similarity_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

def calculate_onske_bonus(row):
    mentor = row["mentor"]
    mentee = row["mentee"]

    mentee_data = mentees_renset[mentees_renset["navn"] == mentee]
    if len(mentee_data) == 0:
        return 0.0

    onske_1 = mentee_data.iloc[0]["onske_1"]
    onske_2 = mentee_data.iloc[0]["onske_2"]
    onske_3 = mentee_data.iloc[0]["onske_3"]

    if mentor == onske_1:
        return 10.0
    if mentor == onske_2:
        return 7.0
    if mentor == onske_3:
        return 4.0
    return 0.0

scored_candidates = all_match_candidates.copy()
scored_candidates["onske_bonus"] = scored_candidates.apply(calculate_onske_bonus, axis=1)
scored_candidates["total_score"] = (
    (scored_candidates["cosine_norm"] * 0.5) +
    (scored_candidates["quality_score_final"] * 0.3) +
    (scored_candidates["onske_bonus"] * 0.2)
).round(2)

print("Total score computed for all mentee-mentor pairs.")


Total score computed for all mentee-mentor pairs.


## 9. Final ordering and export table


In [10]:
import time

openai_client = globals().get("openai_client")
USE_OPENAI_TOP_MATCH_EXPLANATION = True
EXPLAIN_TOP_MATCH_MODEL = "gpt-4.1-mini"

EXPLAIN_TOP_MATCH_SYSTEM = """Write exactly ONE line of prose (a single line in the file: no line breaks, no bullet lists).

Focus almost entirely on concrete academic and professional overlap: fields of study, research themes, tools/methods, industry or role type, seniority/career stage, coursework or project experience—only where the supplied facts or keyword overlap plausibly support it.

Style rules:
- Do NOT start with "The mentee", "The mentor", "The mentee and mentor", "This pairing", or "They". Begin directly with the subject matter (domains, topics, experience).
- Do NOT mention cosine, similarity scores, ranks, decimals, "/10", "0–1", "embedding", or other numeric retrieval metrics. Describe fit in plain words only (e.g. strong / moderate / weak overlap).
- Do NOT mention video, links, or wish-list / preference.
- Optional: at most a few words on whether the written applications feel thin vs detailed—only if it changes how much we can infer about academic fit.

Use only the facts given. Do not invent employers, degrees, or skills not grounded in the overlap or topic headers. Output only that single line."""


def _snippet_label(text, max_len=95):
    t = str(text).replace("\n", " ").strip()
    if len(t) <= max_len:
        return t
    return t[: max_len - 1].rstrip() + "…"


def _profile_alignment_words(row):
    try:
        cn = float(row.get("cosine_norm"))
    except (TypeError, ValueError):
        return "unknown overall profile-text alignment"
    if cn >= 8.2:
        return "strong overall profile-text alignment"
    if cn >= 6.8:
        return "moderate overall profile-text alignment"
    return "limited overall profile-text alignment"


def _retrieval_place_words(row):
    try:
        r = int(row.get("retrieval_rank"))
    except (TypeError, ValueError):
        return "unknown how central this mentor is among all mentors for this mentee's profile text"
    if r <= 3:
        return "among this mentee's closest few mentors by profile-text match"
    if r <= 12:
        return "a mid-ranked mentor for this mentee by profile-text match"
    return "a lower-ranked mentor for this mentee by profile-text match"


def _application_depth_words(row):
    try:
        qo = float(row.get("quality_score_openai"))
    except (TypeError, ValueError):
        return "unknown depth of written application answers"
    if qo >= 7.5:
        return "written answers look fairly detailed"
    if qo >= 5.5:
        return "written answers look moderately detailed"
    return "written answers look quite thin"


def build_match_explanation_facts(row):
    mentee_row = mentees_renset.loc[mentees_renset["navn"] == row["mentee"]]
    if len(mentee_row):
        facts_row = mentee_row.iloc[0]
        q20 = _snippet_label(facts_row.get("question_profile", QUESTION_PROFILE))
        q21 = _snippet_label(facts_row.get("question_motivation", QUESTION_MOTIVATION))
        q23 = _snippet_label(facts_row.get("question_comments", QUESTION_COMMENTS))
    else:
        q20 = _snippet_label(QUESTION_PROFILE)
        q21 = _snippet_label(QUESTION_MOTIVATION)
        q23 = _snippet_label(QUESTION_COMMENTS)
    mt = mentee_text_map.get(row["mentee"], "")
    mo = mentor_text_map.get(row["mentor"], "")
    overlap = find_shared_terms(mt, mo, top_n=8)
    if not str(overlap).strip():
        overlap = "(almost no shared profile keywords detected)"
    return f"""What the three long-form application questions are about (headers only — not the answers):
(1) {q20}
(2) {q21}
(3) {q23}

Retrieval context (wording only — do not repeat as numbers in your answer): {_profile_alignment_words(row)}; {_retrieval_place_words(row)}.

Shared profile keywords (mentee vs mentor combined text): {overlap}

Written-application depth signal (optional, qualitative): {_application_depth_words(row)}.""".strip()


def _finalize_explanation_line(text, max_chars=420):
    line = str(text).strip().split("\n")[0].strip()
    line = " ".join(line.split())
    if not line:
        return line
    if len(line) <= max_chars:
        return line
    cut = line[:max_chars]
    if " " in cut:
        cut = cut[: cut.rfind(" ")].rstrip()
    return cut + "…"


def explain_top_match_openai(row):
    facts = build_match_explanation_facts(row)
    if openai_client is None:
        return ""
    user_prompt = (
        "Write the single-line summary requested in the system message.\n\n"
        f"{facts}"
    )
    last_error = None
    for attempt in range(1, 4):
        try:
            response = openai_client.chat.completions.create(
                model=EXPLAIN_TOP_MATCH_MODEL,
                temperature=0,
                max_tokens=150,
                messages=[
                    {"role": "system", "content": EXPLAIN_TOP_MATCH_SYSTEM},
                    {"role": "user", "content": user_prompt},
                ],
            )
            raw = (response.choices[0].message.content or "").strip()
            return _finalize_explanation_line(raw)
        except Exception as exc:
            last_error = exc
            if attempt < 3:
                time.sleep(1.5 * attempt)
            else:
                return _finalize_explanation_line(f"(explanation failed: {last_error})")


MENTOR_EXPORT_TOP_K = 3
MENTEE_MAX_MENTOR_TOP3_APPEARANCES = 5


def build_mentor_export_trim_then_backfill(full_scored, slots_per_mentor, max_mentee_appearances):
    df = full_scored.sort_values(
        ["mentor", "total_score", "cosine_norm", "similarity_score"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    df = df.drop_duplicates(subset=["mentor", "mentee"], keep="first")
    mentor_order = sorted(df["mentor"].unique())
    mentor_indices = {m: df.index[df["mentor"] == m].tolist() for m in mentor_order}

    picked_set = set()
    for m in mentor_order:
        for i in mentor_indices[m][:slots_per_mentor]:
            picked_set.add(i)
    temp = df.loc[sorted(picked_set)].copy()
    mx = int(temp.groupby("mentee").size().max()) if len(temp) else 0
    print(
        f"Export step 1 — top-{slots_per_mentor}/mentor (mentee uncapped): {len(temp)} rows, "
        f"max mentee appearances = {mx}"
    )

    to_remove = []
    for _, sub in temp.groupby("mentee"):
        if len(sub) <= max_mentee_appearances:
            continue
        drop_idx = (
            sub.sort_values(["total_score", "cosine_norm", "similarity_score"], ascending=[True, True, True])
            .index[: -max_mentee_appearances]
            .tolist()
        )
        to_remove.extend(drop_idx)
    picked_set -= set(to_remove)
    selected = df.loc[sorted(picked_set)].copy()
    print(
        f"Export step 2 — keep best {max_mentee_appearances} rows/mentee by score: "
        f"removed {len(to_remove)}, {len(selected)} rows left"
    )

    for m in mentor_order:
        while (selected["mentor"] == m).sum() < slots_per_mentor:
            mentee_counts = selected.groupby("mentee").size().to_dict()
            mentees_m = set(selected.loc[selected["mentor"] == m, "mentee"].tolist())
            added = False
            for i in mentor_indices[m]:
                if i in selected.index:
                    continue
                u = df.at[i, "mentee"]
                if u in mentees_m:
                    continue
                if mentee_counts.get(u, 0) >= max_mentee_appearances:
                    continue
                selected = pd.concat([selected, df.loc[[i]]])
                added = True
                break
            if not added:
                print(
                    f"Warning: mentor {m!r} has {(selected['mentor'] == m).sum()} row(s) "
                    f"(<{slots_per_mentor}) — no backfill candidate under cap."
                )
                break

    selected = selected.sort_values(
        ["mentor", "total_score", "cosine_norm", "similarity_score"],
        ascending=[True, False, False, False],
    ).reset_index(drop=True)
    over = selected.groupby("mentee").size()
    over = over[over > max_mentee_appearances]
    if len(over):
        print(f"ERROR: mentees still over cap after pipeline: {over.to_dict()}")
    print(
        f"Export step 3 — after backfill: {len(selected)} rows, "
        f"sum total_score = {selected['total_score'].sum():.2f}, "
        f"max mentee count = {int(selected.groupby('mentee').size().max()) if len(selected) else 0}"
    )
    return selected


_export_scored = scored_candidates.copy()
_export_scored["final_rank"] = (
    _export_scored.groupby("mentee")["total_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)
resultater_top3 = build_mentor_export_trim_then_backfill(
    _export_scored,
    slots_per_mentor=MENTOR_EXPORT_TOP_K,
    max_mentee_appearances=MENTEE_MAX_MENTOR_TOP3_APPEARANCES,
)
resultater_top3["mentor_rank"] = (
    resultater_top3.groupby("mentor")["total_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)
resultater_top3["final_rank_score"] = resultater_top3["total_score"]
_mcnt = resultater_top3.groupby("mentor").size()
print(
    f"Mentor export table: {len(resultater_top3)} rows — trim-then-backfill pipeline "
    f"(≤{MENTOR_EXPORT_TOP_K}/mentor; ≤{MENTEE_MAX_MENTOR_TOP3_APPEARANCES}/mentee)."
)
print(
    f"  Mentors with fewer than {MENTOR_EXPORT_TOP_K} rows: {int((_mcnt < MENTOR_EXPORT_TOP_K).sum())}"
)
print(f"  Max mentee appearances across export: {resultater_top3.groupby('mentee').size().max()}")

if USE_OPENAI_TOP_MATCH_EXPLANATION:
    if openai_client is None:
        print("USE_OPENAI_TOP_MATCH_EXPLANATION is True but openai_client is missing; match_explanation left empty.")
        resultater_top3["match_explanation"] = ""
    else:
        print("Generating short OpenAI explanations for exported top-table rows only...")
        resultater_top3["match_explanation"] = resultater_top3.apply(
            lambda row: explain_top_match_openai(row),
            axis=1,
        )
else:
    resultater_top3["match_explanation"] = ""

if USE_OPENAI_SHARED_KEYWORDS and openai_client is not None:
    print(f"Generating OpenAI shared keywords for {len(resultater_top3)} export rows...")
else:
    print("Shared keywords skipped (no OpenAI client).")

resultater_top3["shared_keywords"] = resultater_top3.apply(
    lambda row: find_shared_terms(
        mentee_text_map.get(row["mentee"], ""),
        mentor_text_map.get(row["mentor"], ""),
        top_n=8,
    ),
    axis=1,
)

export_columns = [
    "mentor",
    "mentor_rank",
    "mentee",
    "final_rank_score",
    "total_score",
    "cosine_norm",
    "quality_score_final",
    "quality_score_openai",
    "video_score",
    "onske_bonus",
    "retrieval_rank",
    "shared_keywords",
    "final_rank",
    "match_explanation",
]

resultater_top3 = resultater_top3[export_columns].rename(columns={
    "mentor": "Mentor",
    "mentor_rank": "Mentor Rank",
    "mentee": "Mentee",
    "final_rank_score": "Final Rank Score",
    "total_score": "Total Score",
    "cosine_norm": "Cosine Retrieval Score",
    "quality_score_final": "Quality Score (OpenAI + 10% video)",
    "quality_score_openai": "OpenAI text quality (0–10)",
    "video_score": "Video Score",
    "onske_bonus": "Preference Bonus",
    "retrieval_rank": "Cosine Retrieval Rank",
    "shared_keywords": "Shared keywords (profile overlap)",
    "final_rank": "Final Rank",
    "match_explanation": "Match explanation",
})
resultater_top3 = resultater_top3.sort_values(["Mentor", "Mentor Rank"]).reset_index(drop=True)

print(f"Export ready: {len(resultater_top3)} rows, {resultater_top3['Mentor'].nunique()} mentors.")


Export step 1 — top-3/mentor (mentee uncapped): 342 rows, max mentee appearances = 35
Export step 2 — keep best 5 rows/mentee by score: removed 102, 240 rows left


Export step 3 — after backfill: 342 rows, sum total_score = 1968.08, max mentee count = 5
Mentor export table: 342 rows — trim-then-backfill pipeline (≤3/mentor; ≤5/mentee).
  Mentors with fewer than 3 rows: 0
  Max mentee appearances across export: 5
Generating short OpenAI explanations for exported top-table rows only...


Generating OpenAI shared keywords for 342 export rows...


Export ready: 342 rows, 114 mentors.


## 10. Export Per Mentor


In [11]:
output_path = "/Users/maltejuuljohansen/Desktop/Conflux/Matching_Results.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    resultater_top3.to_excel(writer, sheet_name='Top 3 Mentees Per Mentor', index=False)

print(f"Exported to: {output_path}")

Exported to: /Users/maltejuuljohansen/Desktop/Conflux/Matching_Results.xlsx
